In [2]:
pip install nltk

  Using cached nltk-3.10.0-py3-none-any.whl.metadata (3.2 kB)
  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached regex-2026.7.19-cp314-cp314-win_amd64.whl.metadata (41 kB)
Using cached nltk-3.10.0-py3-none-any.whl (1.7 MB)

   ---------------------------------------- 0/4 [tqdm]
   ---------------------------------------- 0/4 [tqdm]
   ---------------------------------------- 0/4 [tqdm]
   ---------- ----------------------------- 1/4 [regex]
   ---------- ----------------------------- 1/4 [regex]
   -------------------- ------------------- 2/4 [click]
   -------------------- ------------------- 2/4 [click]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ----------------------------

In [3]:
import nltk

In [1]:
import pandas as pd
import numpy as np

import nltk
import re

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Josemila\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Josemila\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [5]:
df = pd.read_excel("Dataset for Data Analytics.xlsx")

df.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [6]:
positive_reviews = [
    "Excellent product and fast delivery",
    "Very good quality highly recommended",
    "Amazing shopping experience",
    "Received the order on time",
    "Satisfied with the purchase",
    "Worth the money",
    "Packaging was excellent",
    "Product quality is outstanding"
]

negative_reviews = [
    "Very poor product quality",
    "Delivery was delayed",
    "Worst shopping experience",
    "Item was damaged",
    "Not worth the money",
    "Very disappointed",
    "Packaging was poor",
    "Customer service is bad"
]

In [7]:
np.random.seed(42)

reviews = []

for status in df["OrderStatus"]:

    if str(status).lower() == "delivered":
        reviews.append(np.random.choice(positive_reviews))

    else:
        reviews.append(np.random.choice(negative_reviews))

df["CustomerReview"] = reviews

In [8]:
df["Sentiment"] = df["OrderStatus"].apply(
    lambda x: "Positive" if str(x).lower()=="delivered" else "Negative"
)

In [9]:
stop_words = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()

In [10]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z]', ' ', text)

    words = text.split()

    words = [word for word in words if word not in stop_words]

    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)

In [11]:
df["CleanReview"] = df["CustomerReview"].apply(clean_text)

df[["CustomerReview","CleanReview"]].head()

,CustomerReview,CleanReview
0,Packaging was poor,packaging poor
1,Item was damaged,item damaged
2,Not worth the money,worth money
3,Packaging was poor,packaging poor
4,Amazing shopping experience,amazing shopping experience


In [12]:
tfidf = TfidfVectorizer(max_features=1000)

X = tfidf.fit_transform(df["CleanReview"])

y = df["Sentiment"]

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [14]:
nb = MultinomialNB()

nb.fit(X_train, y_train)

nb_pred = nb.predict(X_test)

In [15]:
print("Naive Bayes Accuracy")

print(accuracy_score(y_test, nb_pred))

Naive Bayes Accuracy
0.975


In [16]:
print(classification_report(y_test, nb_pred))

              precision    recall  f1-score   support

    Negative       0.97      1.00      0.99       199
    Positive       1.00      0.85      0.92        41

    accuracy                           0.97       240
   macro avg       0.99      0.93      0.95       240
weighted avg       0.98      0.97      0.97       240



In [17]:
svm = LinearSVC()

svm.fit(X_train, y_train)

svm_pred = svm.predict(X_test)

In [18]:
print("SVM Accuracy")

print(accuracy_score(y_test, svm_pred))

SVM Accuracy
0.975


In [19]:
review = ["The product quality is excellent and delivery was fast"]

review_clean = [clean_text(review[0])]

review_vector = tfidf.transform(review_clean)

prediction = svm.predict(review_vector)

print(prediction)

['Positive']


In [20]:
review = ["Worst product and very poor quality"]

review_clean = [clean_text(review[0])]

review_vector = tfidf.transform(review_clean)

prediction = svm.predict(review_vector)

print(prediction)

['Negative']


In [21]:
df.to_excel("Ecommerce_NLP_Output.xlsx", index=False)